In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

from glob import glob

csvs = glob("*.csv")

csvs = [csv for csv in csvs if csv not in ["mnist-vgg-b10.csv", "mnist-vgg-b11.csv", "mnist-vgg-b20.csv"]]

dfs = [pd.read_csv(csv) for csv in csvs]

for i, df in enumerate(dfs):
    df.drop(columns=["Unnamed: 0"], inplace=True, errors='ignore')
    df["dataset"] = csvs[i].split("-")[0]
    df["model"] = csvs[i].split("-")[1]
    df["epoch"] = int(csvs[i].split("-b")[-1].split(".csv")[0])
    df["name"] = csvs[i].split(".csv")[0]
    
data = pd.concat(dfs, ignore_index=True)
data.head()

,Simplification Threshold,Number of Minima,dataset,model,epoch,name
0,0.045368,1,cifar,densenet,143,cifar-densenet-b143
1,0.041440,2,cifar,densenet,143,cifar-densenet-b143
2,0.038883,3,cifar,densenet,143,cifar-densenet-b143
3,0.026006,4,cifar,densenet,143,cifar-densenet-b143
4,0.017306,5,cifar,densenet,143,cifar-densenet-b143


In [2]:
for (model, ds, epoch), df in data.groupby(["model", "dataset", "epoch"]):
	fig = make_subplots(rows=1, cols=1)
	df = df.sort_values(by="Number of Minima", ascending=False, inplace=False)
	name = df["name"].unique()[0]

	fig.add_trace(
		go.Scatter(
			x=df["Simplification Threshold"],
			y=df["Number of Minima"],
			mode="lines",
			name=ds,
			line=dict(shape="hv"),
		),
		row=1,
		col=1,
	)

	fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=600*2, height=300*2, font=dict(size=16))
	fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True)
	fig.update_yaxes(title_text="#Valley Cylinders", type="log", title_standoff=18, automargin=True)
	fig.write_image(f"plots/{name}.png", scale=8)

In [41]:
for model, df in data.groupby("model"):
	dses = df["dataset"].unique()
	fig = make_subplots(rows=1, cols=1, shared_yaxes=True)

	for i, (ds, df) in enumerate(df.groupby(["dataset"])):		
		df = df.sort_values(by="Number of Minima", ascending=False, inplace=False)

		fig.add_trace(
			go.Scatter(
				x=df["Simplification Threshold"],
				y=df["Number of Minima"],
				mode="lines",
				line=dict(shape="hv", color=color_seq[i], width=3),
				name=ds[0],
				opacity=0.6
			),
			row=1,
			col=1
		)
  
		max_x = df["Simplification Threshold"].max()
		if model == "wres":
			max_x = 0.001
		fig.update_xaxes(range=[0, max_x], row=1, col=i+1)
  
	fig.update_annotations(font_size=16)
	fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=14), legend=dict(
     xanchor="right", yanchor="top", x=0.98, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=14)
    ))
	fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True)
	fig.update_yaxes(title_text="#Valley Cylinders", type="log", title_standoff=18, automargin=True)
	fig.write_image(f"plots/{model}.png", scale=8)